In [24]:
import pandas as pd
import pathlib

In [25]:
df_product = pd.read_parquet("../data/raw/shopping_queries_dataset_products.parquet")

In [26]:
df_product.info()

<class 'pandas.DataFrame'>
RangeIndex: 1814924 entries, 0 to 1814923
Data columns (total 7 columns):
 #   Column                Dtype
---  ------                -----
 0   product_id            str  
 1   product_title         str  
 2   product_description   str  
 3   product_bullet_point  str  
 4   product_brand         str  
 5   product_color         str  
 6   product_locale        str  
dtypes: str(7)
memory usage: 2.1 GB


In [27]:
df_product.isnull().sum()

product_id                   0
product_title                0
product_description     877799
product_bullet_point    304116
product_brand           142756
product_color           691001
product_locale               0
dtype: int64

In [28]:
(df_product['product_locale'].unique())

<ArrowStringArray>
['es', 'us', 'jp']
Length: 3, dtype: str

In [29]:
df_us_product = df_product[df_product['product_locale'] =='us'].copy()

In [30]:
df_us_product.info()

<class 'pandas.DataFrame'>
Index: 1215854 entries, 167168 to 1753351
Data columns (total 7 columns):
 #   Column                Non-Null Count    Dtype
---  ------                --------------    -----
 0   product_id            1215854 non-null  str  
 1   product_title         1215854 non-null  str  
 2   product_description   646062 non-null   str  
 3   product_bullet_point  1037078 non-null  str  
 4   product_brand         1143519 non-null  str  
 5   product_color         809750 non-null   str  
 6   product_locale        1215854 non-null  str  
dtypes: str(7)
memory usage: 1.4 GB


In [31]:
df_us_product.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,NaN,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,NaN,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,NaN,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
167171,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,NaN,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,NaN,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [32]:
df_us_product.isnull().sum()

product_id                   0
product_title                0
product_description     569792
product_bullet_point    178776
product_brand            72335
product_color           406104
product_locale               0
dtype: int64

In [33]:
df_us_product['product_id'].duplicated().sum()

np.int64(0)

In [34]:
# Fill missing values to prevent 'Nan' appearing in index
for col in ['product_bullet_point', 'product_brand', 'product_color', 'product_description']:
    df_us_product[col] = df_us_product[col].fillna('')

In [35]:
df_us_product.isnull().sum()

product_id              0
product_title           0
product_description     0
product_bullet_point    0
product_brand           0
product_color           0
product_locale          0
dtype: int64

In [36]:
for col in ['product_title', 'product_brand', 'product_color', 'product_bullet_point']:
    df_us_product[col] = df_us_product[col].str.replace(r'\s+', ' ', regex=True).str.strip()

In [37]:
df_us_product['product_text'] = (
    "Title: " + df_us_product['product_title'] +
    " | Brand: " + df_us_product['product_brand'] +
    " | Color: " + df_us_product['product_color'] +
    " | Specs: " + df_us_product['product_bullet_point']
)
df_us_product['product_text'] = df_us_product['product_text'].str.replace(r'[\t\n\r]+', ' ',regex=True)

In [38]:
df_us_product.info()

<class 'pandas.DataFrame'>
Index: 1215854 entries, 167168 to 1753351
Data columns (total 8 columns):
 #   Column                Non-Null Count    Dtype
---  ------                --------------    -----
 0   product_id            1215854 non-null  str  
 1   product_title         1215854 non-null  str  
 2   product_description   1215854 non-null  str  
 3   product_bullet_point  1215854 non-null  str  
 4   product_brand         1215854 non-null  str  
 5   product_color         1215854 non-null  str  
 6   product_locale        1215854 non-null  str  
 7   product_text          1215854 non-null  str  
dtypes: str(8)
memory usage: 2.2 GB


In [39]:
df_us_product.head(3)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,product_text
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,,Virtually silent at less than 0.3 sones Precis...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us,Title: Delta BreezSignature VFB25ACH 80 CFM Ex...
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us,Title: Aero Pure AP80RVLW Super Quiet 80 CFM R...
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us,Title: Aero Pure AP120H-SL W Slim Fit 120 CFM ...


In [40]:
df_us_product['product_text'] = df_us_product['product_text'].str.replace(r'[\t\n\r]+', ' ', regex=True)
df_us_product = df_us_product.drop(columns=['product_locale'])
df_us_product.columns

Index(['product_id', 'product_title', 'product_description',
       'product_bullet_point', 'product_brand', 'product_color',
       'product_text'],
      dtype='str')

In [41]:
# Save the dataframe to SQLite (using half the dataset)

from sqlalchemy import create_engine, types

# Create engine
engine = create_engine('sqlite:///../data/processed/products_dataset.db')

# Reset the index
df_product_save = df_us_product.reset_index(drop=True)
df_product_save.index.name = 'pid'

# Use only half the dataset
half = len(df_product_save) // 3
df_product_save = df_product_save.iloc[:half]

# Stream to SQLite
df_product_save.to_sql(
    name = 'products',
    con = engine,
    if_exists = 'replace',
    index = True,
    chunksize = 10000,
    dtype={'pid': types.Integer()}
)

405284

In [42]:
# Export the tsv file

import sqlite3

conn = sqlite3.connect('../data/processed/products_dataset.db')

query = 'SELECT pid, product_text FROM products'
df_product_final = pd.read_sql_query(query, conn)

# Export the tsv format
df_product_final.to_csv('collection.tsv', sep='\t', header=False, index=False)

conn.close()

In [43]:
df_product_final.head(3)

,pid,product_text
0,0,Title: Delta BreezSignature VFB25ACH 80 CFM Ex...
1,1,Title: Aero Pure AP80RVLW Super Quiet 80 CFM R...
2,2,Title: Aero Pure AP120H-SL W Slim Fit 120 CFM ...


In [44]:
df_product_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 405284 entries, 0 to 405283
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   pid           405284 non-null  int64
 1   product_text  405284 non-null  str  
dtypes: int64(1), str(1)
memory usage: 274.9 MB
